# A second school-level factor: the maths-and-science versus English-and-open "tilt"

`gcse-school-quality.ipynb` models each school's six GCSE value-added elements with a **general quality** $g_i$ and a **consistency** $s_i$. Its residual check found structure that one general factor does not capture:

- Maths-Science and English-Open correlate about 0.05 *more* than the one-factor model implies, and the cross pairs (Maths-Open, English-Science) about 0.04 *less*.
- Refitting without Open changed which pair defines $g_i$, and it removed the link between quality and consistency ($\rho$).

This notebook adds a **second factor**, a school's *tilt* $h_i$:

$$
x_{ie} = \mu_e + \lambda_e\, g_i + \kappa_e\, h_i + \delta_{ie}, \qquad \delta_{ie} \sim \text{Normal}(0,\ \tau_e\, s_i), \qquad h_i \sim \text{Normal}(0, 1)
$$

with the same consistency structure as before, $\log s_i = \sigma_s(\rho\, g_i + \sqrt{1-\rho^2}\, w_i)$, and the same known measurement error. A school with positive $h_i$ is stronger in the elements with positive $\kappa_e$ than its general quality predicts, and weaker in those with negative $\kappa_e$.

**We do not choose the groups.** The one-factor pattern suggests Maths and Science load one way and English and Open the other, but that was read off the same data. So $\kappa_e$ is estimated freely for every element, with two constraints to make it identifiable:

- **Humanities is the anchor:** $\kappa_{Humanities} = 0$. Two orthogonal factors can be rotated into each other; fixing one loading removes that freedom. Humanities is chosen because it sat in the middle of the pattern (its correlations with everything were close to the one-factor prediction).
- **The sign of $h_i$ is fixed** by requiring $\kappa_{Maths} > 0$.

Whatever pattern appears is then the data's, subject to the choice of anchor. We fit the earlier model (same schools, same priors) alongside so the two can be compared directly.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
from scipy.stats import spearmanr

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")
print(f"Running on PyMC v{pm.__version__}")

## Data

Exactly as in `gcse-school-quality.ipynb`: one row per school and GCSE element, with the standard error recovered from the published confidence interval, keeping schools with at least three elements.

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")

elements = ["English", "Maths", "Science", "Humanities", "Languages", "Open"]
column = {"English": "P8MEAENG", "Maths": "P8MEAMAT", "Science": "SCIVAMEA_PTQ_EE",
          "Humanities": "HUMVAMEA_PTQ_EE", "Languages": "LANVAMEA_PTQ_EE", "Open": "P8MEAOPEN"}

frames = []
for e in elements:
    c = column[e]
    sub = raw[["URN", c, f"{c} lower", f"{c} upper"]].dropna()
    sub.columns = ["URN", "va", "lower", "upper"]
    sub["element"] = e
    frames.append(sub)
long = pd.concat(frames)
long["se"] = (long["upper"] - long["lower"]) / (2 * 1.96)

n_elements_per_school = long.groupby("URN")["element"].nunique()
long = long[long["URN"].isin(n_elements_per_school[n_elements_per_school >= 3].index)].sort_values("URN").reset_index(drop=True)

urns = pd.Index(sorted(long["URN"].unique()))
long["school_idx"] = urns.get_indexer(long["URN"])
long["element_idx"] = long["element"].map({e: k for k, e in enumerate(elements)}).to_numpy()

n_schools, n_elements = len(urns), len(elements)
x_obs, x_se = long["va"].to_numpy(), long["se"].to_numpy()
s_idx, e_idx = long["school_idx"].to_numpy(), long["element_idx"].to_numpy()
counts = np.bincount(s_idx, minlength=n_schools)
print(f"{n_schools} schools, {len(long)} school-element observations")

## Models

**Baseline:** the varying-consistency model from `gcse-school-quality.ipynb`. **Tilt:** the same model with the second factor added. In both, the element-specific departures $\delta_{ie}$ are marginalised out, as before, so each observation is Normal with variance $\tau_e^2 s_i^2 + \sigma_{ie}^2$.

In the tilt model, $\kappa_e$ has a Normal(0, 0.5) prior for English, Science, Languages and Open, is fixed at 0 for Humanities, and is the absolute value of a Normal(0, 0.5) for Maths, i.e. a HalfNormal(0.5) (the sign convention). The loadings are built with a mask rather than by stacking separate scalars, because the stacked form made the sampler's compilation step run out of memory.

In [ ]:
is_maths = np.array([e == "Maths" for e in elements])
keep = np.array([0.0 if e == "Humanities" else 1.0 for e in elements])

def build_model(tilt):
    with pm.Model(coords={"element": elements}) as model:
        mu = pm.Normal("mu", 0, 1, dims="element")
        lam = pm.HalfNormal("lam", 1, dims="element")
        tau = pm.HalfNormal("tau", 0.5, dims="element")
        g = pm.Normal("g", 0, 1, shape=n_schools)
        sigma_s = pm.HalfNormal("sigma_s", 0.5)
        rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
        w = pm.Normal("w", 0, 1, shape=n_schools)
        log_s = pm.Deterministic("log_s", sigma_s * (rho * g + pt.sqrt(1 - rho**2) * w))
        mean = mu[e_idx] + lam[e_idx] * g[s_idx]
        if tilt:
            h = pm.Normal("h", 0, 1, shape=n_schools)
            k_raw = pm.Normal("kappa_raw", 0, 0.5, shape=n_elements)
            # Maths is forced positive (sign convention for h); Humanities is fixed at 0 (rotation anchor)
            kappa = pm.Deterministic("kappa", pt.where(is_maths, pt.abs(k_raw), k_raw) * keep, dims="element")
            mean = mean + kappa[e_idx] * h[s_idx]
        pm.Normal("x_obs", mu=mean, sigma=pt.sqrt((tau[e_idx] * pt.exp(log_s[s_idx]))**2 + x_se**2), observed=x_obs)
    return model

baseline_model = build_model(tilt=False)
tilt_model = build_model(tilt=True)

### Fit

In [ ]:
with baseline_model:
    idata_base = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)

In [ ]:
with tilt_model:
    idata_tilt = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)

### Diagnostics

In [ ]:
for name, idata in [("baseline", idata_base), ("tilt", idata_tilt)]:
    print(f"{name}: divergences = {int(idata.sample_stats['diverging'].sum())}")
summary = az.summary(idata_tilt, var_names=["mu", "lam", "tau", "kappa", "sigma_s", "rho"], round_to=3)
print(f"tilt model: worst r_hat = {summary['r_hat'].max():.3f}, smallest bulk ESS = {summary['ess_bulk'].min():.0f}, smallest tail ESS = {summary['ess_tail'].min():.0f}")
summary.loc[[i for i in summary.index if i.startswith(("kappa", "sigma_s", "rho"))]]

rh, es = az.rhat(idata_tilt), az.ess(idata_tilt)
skip = {"kappa_raw", "rho_raw"}     # kappa_raw[Maths] enters only through its absolute value, so its sign is arbitrary
worst = pd.DataFrame({"max r_hat": {v: float(rh[v].max()) for v in rh.data_vars if v not in skip},
                      "min bulk ESS": {v: float(es[v].min()) for v in es.data_vars if v not in skip}}).sort_values("max r_hat", ascending=False)
display(worst.round(3))
tau_sci = idata_tilt.posterior["tau"].sel(element="Science").to_numpy()
print("tau[Science] chain means:", tau_sci.mean(axis=1).round(3), "| 1%, 5%, 50%, 95%, 99% quantiles:", np.percentile(tau_sci, [1, 5, 50, 95, 99]).round(3))
print("baseline tau[Science] mean:", idata_base.posterior["tau"].sel(element="Science").to_numpy().mean().round(3))

**Reading the diagnostics.** The baseline samples cleanly. The tilt model has no divergences, but its sampling is poorer: $\hat R$ reaches 1.15 for $\tau$ and 1.09 for $\kappa$, and the smallest ESS is 19, against 200 or more for everything in the baseline. The cause is one parameter, $\tau_{Science}$. The chains agree on every other parameter, with chain means within about 0.005 of each other (posterior SDs are about 0.01), but $\tau_{Science}$ has chain means from 0.06 to 0.08 and a posterior that runs from near zero up to about 0.12. The tilt has absorbed almost all of Science's specific scatter, so the likelihood is almost flat as $\tau_{Science}$ approaches zero, and NUTS explores such a long, flat tail slowly. So treat $\tau_{Science}$ as poorly determined; $\kappa$, $\lambda$, $\sigma_s$ and $\rho$ are reliable to the accuracy quoted. ($\kappa_{raw}$ is left out of the table: Maths enters only through its absolute value, so the sign of its raw parameter is arbitrary.)

## What does the second factor look like?

Left: the loadings $\kappa_e$ on the tilt, with 89% intervals. Right: the loadings $\lambda_e$ on general quality, for the baseline and the tilt model. If the tilt is the maths-and-science versus English-and-open split, $\kappa$ should be positive for Maths and Science and negative for English and Open, with Humanities (fixed) and Languages near zero.

In [ ]:
post_b, post_t = idata_base.posterior, idata_tilt.posterior
kappa_d = post_t["kappa"].to_numpy().reshape(-1, n_elements)
lam_b = post_b["lam"].to_numpy().reshape(-1, n_elements)
lam_t = post_t["lam"].to_numpy().reshape(-1, n_elements)

def interval(x): return np.percentile(x, [5.5, 50, 94.5], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
lo, med, hi = interval(kappa_d)
for k in range(n_elements):
    axes[0].plot([lo[k], hi[k]], [k, k], color="#4C72B0", linewidth=2)
    axes[0].plot(med[k], k, "o", color="#4C72B0")
axes[0].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("loading on tilt, $\\kappa_e$ (Humanities fixed at 0)")
for shift, d, label, colour in [(-0.12, lam_b, "baseline", "#4C72B0"), (0.12, lam_t, "with tilt", "#DD8452")]:
    lo, med, hi = interval(d)
    for k in range(n_elements):
        axes[1].plot([lo[k], hi[k]], [k + shift, k + shift], color=colour, linewidth=2, label=label if k == 0 else None)
        axes[1].plot(med[k], k + shift, "o", color=colour)
axes[1].set_xlabel("loading on general quality, $\\lambda_e$")
axes[1].legend()
axes[0].set_yticks(range(n_elements), elements)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

pd.DataFrame({"kappa": kappa_d.mean(axis=0), "P(kappa > 0)": (kappa_d > 0).mean(axis=0), "lam baseline": lam_b.mean(axis=0), "lam tilt": lam_t.mean(axis=0),
              "tau baseline": post_b["tau"].to_numpy().reshape(-1, n_elements).mean(axis=0), "tau tilt": post_t["tau"].to_numpy().reshape(-1, n_elements).mean(axis=0)},
             index=elements).round(3)

## Is the residual structure gone?

For each posterior draw we have a complete set of school parameters, so every school and element has a standardised residual $r_{ie} = (x^{obs}_{ie} - \mu_e - \lambda_e g_i - \kappa_e h_i)/\sqrt{\tau_e^2 s_i^2 + \sigma_{ie}^2}$. If the model is right, these are independent Normal(0, 1) numbers, so their correlations across schools should be zero apart from sampling noise (about $\pm 0.03$ with 1,808 schools). We compute the correlation of the residuals over the schools with all six elements in each draw, average over draws, and compare with the same statistic on data replicated from the model (which is zero by construction, with only the sampling spread). This is a stricter check than the one in `gcse-school-quality.ipynb`, which correlated residuals averaged over draws; averaging builds in negative correlation among elements that share a latent factor.

In [ ]:
thin = slice(0, 1000, 5)   # 200 draws per chain, 800 in total
def draws(post, name):
    a = post[name].to_numpy()[:, thin]
    return a.reshape(-1, *a.shape[2:])

full6 = counts == 6
six_rows = full6[s_idx]
order6 = np.lexsort((e_idx[six_rows], s_idx[six_rows]))    # school by school, elements in a fixed order

def realized_corr(post, tilt):
    mu_d, lam_d, tau_d, g_d, ls_d = (draws(post, n) for n in ["mu", "lam", "tau", "g", "log_s"])
    mean = mu_d[:, e_idx] + lam_d[:, e_idx] * g_d[:, s_idx]
    if tilt:
        mean = mean + draws(post, "kappa")[:, e_idx] * draws(post, "h")[:, s_idx]
    sd = np.sqrt((tau_d[:, e_idx] * np.exp(ls_d[:, s_idx]))**2 + x_se**2)
    out = {}
    for label, r in [("observed", (x_obs[None, :] - mean) / sd), ("replicated", rng.standard_normal(mean.shape))]:
        r6 = r[:, six_rows][:, order6].reshape(r.shape[0], -1, n_elements)
        out[label] = np.stack([np.corrcoef(r6[d].T) for d in range(r6.shape[0])])
    return out

res_b, res_t = realized_corr(post_b, False), realized_corr(post_t, True)

fig, axes = plt.subplots(1, 3, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1, 0.04]})   # third column holds the colorbar
for ax, res, title in [(axes[0], res_b, "baseline"), (axes[1], res_t, "with tilt")]:
    corr = res["observed"].mean(axis=0)
    im = ax.imshow(corr, vmin=-0.3, vmax=0.3, cmap="RdBu_r")
    ax.set_xticks(range(6), elements, rotation=30); ax.set_yticks(range(6), elements)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(f"Residual correlation, {title}")
    ax.grid(False)
fig.colorbar(im, cax=axes[2])
plt.show()

ix = {e: k for k, e in enumerate(elements)}
pairs = [("Maths", "Science"), ("English", "Open"), ("Maths", "Open"), ("English", "Science"), ("English", "Maths"), ("Science", "Open"),
         ("Humanities", "English"), ("Humanities", "Maths"), ("Languages", "English")]
rows = []
for a, b in pairs:
    i, j = ix[a], ix[b]
    row = {"pair": f"{a}-{b}"}
    for name, res in [("baseline", res_b), ("with tilt", res_t)]:
        rep = res["replicated"][:, i, j]
        row[f"{name}: observed"] = res["observed"][:, i, j].mean()
        row[f"{name}: replicated 89%"] = f"[{np.percentile(rep, 5.5):.3f}, {np.percentile(rep, 94.5):.3f}]"
    rows.append(row)
print(f"{full6.sum()} schools with all six elements")
pd.DataFrame(rows).set_index("pair").round(3)

## Does anything else change?

We compare, on the same schools: general quality $g_i$, consistency $\log s_i$, and the population parameters $\sigma_s$ and $\rho$. The size of $g_i$ has no natural unit and its overall level trades off against $\mu_e$, so we compare the identified **average expected value added**, $\bar\mu + \bar\lambda\, g_i$.

In [ ]:
g_b = post_b["g"].to_numpy().reshape(-1, n_schools).mean(axis=0)
g_t = post_t["g"].to_numpy().reshape(-1, n_schools).mean(axis=0)
h_t = post_t["h"].to_numpy().reshape(-1, n_schools)
h_mean, h_sd = h_t.mean(axis=0), h_t.std(axis=0)
ls_b = post_b["log_s"].to_numpy().reshape(-1, n_schools).mean(axis=0)
ls_t = post_t["log_s"].to_numpy().reshape(-1, n_schools).mean(axis=0)
A_b = post_b["mu"].to_numpy().mean() + lam_b.mean() * g_b
A_t = post_t["mu"].to_numpy().mean() + lam_t.mean() * g_t
full6 = counts == 6
k = int(0.1 * n_schools)
def overlap(x, y, largest):
    pick = (lambda v: set(np.argsort(-v)[:k])) if largest else (lambda v: set(np.argsort(v)[:k]))
    return len(pick(x) & pick(y)) / k

def summarise(x):
    lo, hi = np.percentile(x, [5.5, 94.5])
    return f"mean={x.mean():.3f}, 89% interval [{lo:.3f}, {hi:.3f}]"

print(f"average expected VA:  correlation {np.corrcoef(A_b, A_t)[0, 1]:.3f}, rank correlation {spearmanr(A_b, A_t)[0]:.3f}; "
      f"top tenth overlap {overlap(A_b, A_t, True):.2f}, bottom tenth {overlap(A_b, A_t, False):.2f}")
print(f"log consistency:      correlation {np.corrcoef(ls_b, ls_t)[0, 1]:.3f}, rank correlation {spearmanr(ls_b, ls_t)[0]:.3f}; "
      f"most consistent tenth overlap {overlap(ls_b, ls_t, False):.2f}")
print("sigma_s: baseline", summarise(post_b["sigma_s"].to_numpy().ravel()), "\n         tilt    ", summarise(post_t["sigma_s"].to_numpy().ravel()))
print("rho:     baseline", summarise(post_b["rho"].to_numpy().ravel()), "\n         tilt    ", summarise(post_t["rho"].to_numpy().ravel()))
print(f"tilt h: posterior SD median {np.median(h_sd):.2f} for six-element schools {np.median(h_sd[full6]):.2f} (prior 1); correlation of posterior means, h vs g: {np.corrcoef(h_mean, g_t)[0, 1]:.3f}, h vs log s: {np.corrcoef(h_mean, ls_t)[0, 1]:.3f}")

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
axes[0].scatter(A_b, A_t, s=5, alpha=0.4, color="#4C72B0"); axes[0].set_xlabel("average expected VA, baseline"); axes[0].set_ylabel("with tilt")
axes[1].scatter(ls_b, ls_t, s=5, alpha=0.4, color="#DD8452"); axes[1].set_xlabel("$\\log s_i$, baseline"); axes[1].set_ylabel("with tilt")
for ax, lim in [(axes[0], (A_b.min(), A_b.max())), (axes[1], (ls_b.min(), ls_b.max()))]:
    ax.plot(lim, lim, color="grey", linewidth=0.8, linestyle="--")
for ax, name, label in [(axes[2], "sigma_s", "$\\sigma_s$"), (axes[3], "rho", "$\\rho$")]:
    ax.hist(post_b[name].to_numpy().ravel(), bins=40, density=True, alpha=0.6, color="#4C72B0", label="baseline")
    ax.hist(post_t[name].to_numpy().ravel(), bins=40, density=True, alpha=0.6, color="#DD8452", label="with tilt")
    ax.set_xlabel(label)
axes[3].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[2].legend()
plt.tight_layout()
plt.show()

## Where do schools sit on the tilt?

The tilt is only informative if it separates schools. Left: the posterior mean of $h_i$ against general quality, coloured by number of elements. Right: the mean tilt by region (descriptive; each school's $h_i$ is shrunk to zero nationally, not to its region's mean).

In [ ]:
region = pd.Series(urns.map(raw.drop_duplicates("URN").set_index("URN")["RGN24NM"]), index=range(n_schools))
by_region = pd.DataFrame({"h": h_mean, "g": g_t, "region": region}).groupby("region").agg(schools=("h", "count"), mean_h=("h", "mean"), sd_h=("h", "std"), mean_g=("g", "mean")).sort_values("mean_h", ascending=False)
display(by_region.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
sc = axes[0].scatter(g_t, h_mean, c=counts, cmap="viridis", s=8, alpha=0.6)
axes[0].axhline(0, color="grey", linewidth=0.8, linestyle="--"); axes[0].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("general quality $g_i$ (posterior mean)"); axes[0].set_ylabel("tilt $h_i$ (posterior mean)")
plt.colorbar(sc, ax=axes[0], label="number of GCSE elements")
order = list(by_region.index)
axes[1].boxplot([h_mean[(region == r_).to_numpy()] for r_ in order], orientation="horizontal", tick_labels=order, showfliers=False)
axes[1].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("tilt $h_i$ (posterior mean)")
axes[1].invert_yaxis()
plt.show()

## Posterior predictive check

The same tail check as in the first notebook, $T_i = \frac{1}{k_i}\sum_e r_{ie}^2$, comparing the share of schools above each threshold in the data and in data replicated from each model.

In [ ]:
import scipy.sparse as sp
school_matrix = sp.csr_matrix((np.ones(len(s_idx)), (s_idx, np.arange(len(s_idx)))), shape=(n_schools, len(s_idx)))

def ppc_share(post, tilt, thresholds=(1.5, 2.0, 3.0)):
    mu_d, lam_d, tau_d, g_d, ls_d = (draws(post, n) for n in ["mu", "lam", "tau", "g", "log_s"])
    mean = mu_d[:, e_idx] + lam_d[:, e_idx] * g_d[:, s_idx]
    if tilt:
        mean = mean + draws(post, "kappa")[:, e_idx] * draws(post, "h")[:, s_idx]
    sd = np.sqrt((tau_d[:, e_idx] * np.exp(ls_d[:, s_idx]))**2 + x_se**2)
    rep = mean + sd * rng.standard_normal(mean.shape)
    out = {}
    for label, values in [("observed", x_obs[None, :]), ("replicated", rep)]:
        T = (school_matrix @ (((values - mean) / sd) ** 2).T).T / counts
        out[label] = np.stack([(T > t).mean(axis=1) for t in thresholds], axis=1)
    return out

rows = []
for name, p_, tilt in [("baseline", post_b, False), ("with tilt", post_t, True)]:
    res = ppc_share(p_, tilt)
    for j, t in enumerate((1.5, 2.0, 3.0)):
        lo, hi = np.percentile(res["replicated"][:, j], [5.5, 94.5])
        rows.append({"model": name, "threshold_T": t, "observed_share": res["observed"][:, j].mean(),
                     "replicated_mean": res["replicated"][:, j].mean(), "replicated_89%": f"[{lo:.3f}, {hi:.3f}]"})
pd.DataFrame(rows).round(3)

## Summary

**The second factor is real, and it is the split the residuals suggested.** Estimated with no groups imposed (only Humanities as the anchor and the sign of Maths fixed), the tilt loads positively on Maths ($\kappa = +0.105$) and Science ($+0.154$), negatively on English ($-0.077$) and Open ($-0.079$), and slightly negatively on Languages ($-0.038$, interval [-0.071, -0.005]). English and Open come out almost equal, which is what a shared "language and expressive" side would give. A school with a high tilt is stronger in Maths and Science than its general quality predicts and weaker in English and Open.

- **It repairs the misfit.** Per-draw residual correlations in the baseline were $+0.27$ (Maths-Science) and $+0.26$ (English-Open) against $\pm 0.04$ expected, with the cross pairs at $-0.13$ to $-0.16$. With the tilt they are $-0.00$ and $+0.05$, and the cross pairs $-0.02$. Small leftovers remain, a few just outside the null range of about $\pm 0.04$ (English-Maths $+0.07$, English-Open $+0.05$, Humanities with English and Maths about $-0.06$, Languages-English $-0.04$; the Humanities ones are probably an effect of the anchor), and the extreme-tail posterior predictive check is unchanged, so the model is better, not perfect.
- **It is modest in size but is the main systematic part of what general quality leaves.** One SD of tilt is worth about 0.08-0.15 value-added points per element, against about 0.5 per SD of general quality, so it explains roughly 2-8% of the between-school variance in these elements. But the element-specific *variance* ($\tau^2$) in English, Maths and Open falls by about a quarter to two fifths (English $\tau$ 0.175 to 0.138, Maths 0.173 to 0.152, Open 0.193 to 0.154, i.e. variance down 38%, 23% and 36%) and by most of it for Science ($\tau$ 0.166 to about 0.08, poorly determined, see the diagnostics). So a good part of what looked like idiosyncratic element scatter is systematic tilt.
- **General quality does not change.** The average expected value added correlates 0.999 with the baseline and 96% of the top and bottom tenths stay put. The concern from the drop-Open check, that the pairing tilts what $g_i$ means, does not affect the ranking once the tilt is modelled.
- **The tilt is unrelated to quality and to consistency.** The correlation of posterior means of $h_i$ with $g_i$ is 0.04 and with $\log s_i$ is 0.04, so it is a separate dimension: two schools of equal general quality can differ in which side they lean to.
- **Consistency and $\rho$ survive, and $\rho$ strengthens.** $\sigma_s$ rises from 0.355 to 0.388 [0.352, 0.422] and $\rho$ from $-0.19$ to $-0.26$ [-0.33, -0.19]. In `gcse-school-quality.ipynb` $\rho$ depended on Open being in the model; here Open stays in and the pairing is modelled, and the link between quality and consistency is, if anything, clearer. The ranking of consistency does move (rank correlation 0.88 with the baseline; 65% of the most consistent tenth stays), because part of what looked like inconsistency was tilt.
- **A school's tilt is only moderately well determined.** The posterior SD of $h_i$ is about 0.63 against a prior of 1, so it is a weak signal for one school, better used through its population structure than as a ranking.
- **Regions differ a little.** Mean tilt is highest in the South West (+0.21) and South East (+0.15) and lowest in the West Midlands (-0.24), against a spread of about 0.75 across schools; descriptive, and small in value-added terms (a tilt of 0.2 is about 0.02 points in Maths).

**Caveats.** $\kappa_e$ for the other elements is defined relative to Humanities, which is fixed at zero; a different anchor would rotate the factors, though not the fit. $\tau_{Science}$ is poorly determined (see the diagnostics). One year of data, association not cause. On the interpretation: English and Open sit together on the negative side, so the tilt is not an Open-specific arts dimension; Open shares the "language and expressive" side with English, and Maths and Science the other. Whether that reflects breadth of curriculum, subject choice, or how the Open bucket is filled cannot be separated with these data.

**Next.** Carry $g_i$, $h_i$ and $s_i$ (and the regional layer) into the A-level model, where the tilt gives a natural test: does a school's Maths-Science versus English-Open tilt at GCSE predict the corresponding pattern at A-level, beyond general quality?